# 🗄️ تحليل `core/cache.py` — نظام الكاش (Sprint 5، الخطوة 1)

---

## 🎯 TL;DR

> **4 مكونات**: `BaseCache` (العقد) + `MemoryCache` (محلي) + `RedisCache` (خادم) + `get_cache()` (Factory). **الفكرة**: كاش موحّد يقلل استهلاك الـ API، مع **fallback تلقائي** من Redis إلى Memory عند الفشل.

---

## 📊 المخطط المعماري

```mermaid
graph TD
    A[BaseCache ABC] --> B[MemoryCache]
    A --> C[RedisCache]
    D[get_cache factory] --> E{Config backend}
    E -->|memory| B
    E -->|redis| F{ping OK?}
    F -->|Yes| C
    F -->|No| B

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style B fill:#bfb,stroke:#333,stroke-width:2px
    style C fill:#f9f,stroke:#333,stroke-width:2px
    style D fill:#bdf,stroke:#333,stroke-width:2px
```

---

## 🔬 التشريح — 4 مكونات

### 1️⃣ `BaseCache` — العقد (4 دوال)

| الدالة | التوقيع | الدور |
|--------|---------|-------|
| `get(key)` | `-> Any \| None` | جلب قيمة |
| `set(key, value, ttl=60)` | `-> None` | تخزين + صلاحية |
| `clear()` | `-> None` | مسح الكل |
| `stats()` | `-> dict` | إحصائيات |

**🎯 4 دوال فقط** — YAGNI في أنقى صوره. لا `delete(key)`, لا `exists()` — يمكن إضافتها لاحقاً.

---

### 2️⃣ `MemoryCache` — قاموس بسيط

```python
self.store: dict[str, tuple[Any, float]] = {}
# key → (value, expires_at)
```

**دورة حياة المفتاح:**

```mermaid
graph TD
    A[set key, value, ttl] --> B[expires_at = now + ttl]
    B --> C[store key = value, expires_at]
    
    D[get key] --> E{key in store?}
    E -->|No| F[misses++]
    E -->|Yes| G{now > expires_at?}
    G -->|Yes| H[delete key + misses++]
    G -->|No| I[hits++ + return value]

    style C fill:#bfb,stroke:#333
    style I fill:#bfb,stroke:#333
    style F fill:#fdd,stroke:#333
    style H fill:#fdd,stroke:#333
```

**🎯 تريكة `(value, expire_time)`:** بدلاً من كلاس `CacheEntry` منفصل — tuple مباشر. **بساطة + أداء**.

**🎯 تريكة الحذف الكسول (Lazy Deletion):** لا حذف تلقائي — `get` يكتشف الانتهاء ويحذف. لا حاجة لـ background thread.

**🎯 إحصائيات:** `hits` + `misses` + `items_count` — لحساب **Hit Ratio** لاحقاً.

---

### 3️⃣ `RedisCache` — خادم خارجي

**🎯 تريكة `import redis` داخل `__init__`:**
```python
def __init__(self, ...):
    import redis   # ← هنا فقط
```
**السبب:** لو `redis` غير مثبتة، `MemoryCache` يعمل بدون مشاكل. الاستيراد في المستوى الأعلى كان سيكسر التطبيق كله.

**🎯 تريكة `self.client.ping()`:** فحص الاتصال فوراً. لو Redis غير متاح → exception → `get_cache` يلتقطه ويستخدم Memory.

**Serialization:**

| النوع | المعالجة |
|-------|----------|
| `dict`, `list` | `json.dumps(value)` |
| غير ذلك | `str(value)` |
| القراءة | `json.loads()` مع fallback للنص |

**🎯 لماذا `json`؟** Redis يخزن **strings/bytes** فقط. `Decimal` و `datetime` يحتاجان تحويلاً — لكن هنا نتعامل مع dict/list بسيطة.

**🎯 `setex(key, seconds, value)`:** أمر Redis ذري — يضبط TTL تلقائياً.

**🎯 `flushdb()` في `clear()`:** يمسح **قاعدة البيانات الحالية فقط** (وليس كل Redis).

**🎯 Fail-Safe:** كل عملية داخل `try/except` — لو Redis سقط، الكاش **لا يُسقط التطبيق**.

---

### 4️⃣ `get_cache()` — Factory

```mermaid
graph TD
    A[Config.get cache] --> B{backend?}
    B -->|memory| C[MemoryCache]
    B -->|redis| D[try RedisCache]
    D -->|OK| E[RedisCache]
    D -->|Fail| F[logger.warning]
    F --> C

    style E fill:#bfb,stroke:#333
    style C fill:#bfb,stroke:#333
    style F fill:#fdd,stroke:#333
```

**الإعدادات من `config.json`:**
```json
{
  "cache": {
    "backend": "memory",
    "default_ttl": 60,
    "redis_url": "redis://localhost:6379/0"
  }
}
```

**🎯 تريكة Fallback:** `redis` فشل → Memory. **Zero Downtime** — التطبيق لا يتوقف أبداً.

---

## 📊 مقارنة Memory vs Redis

| البعد | MemoryCache | RedisCache |
|-------|-------------|------------|
| **السرعة** | ⚡ فوري (ns) | 🌐 شبكة (ms) |
| **الحجم** | محدود بالذاكرة | GBs |
| **المشاركة** | عملية واحدة | عدة عمليات/خوادم |
| **الاستمرارية** | ❌ (RAM) | ✅ (اختياري) |
| **التثبيت** | مدمج | يحتاج `pip install redis` + خادم |
| **الاستخدام** | CLI محلي | Production متعدد |

---

## 🧪 اختبار (وصف)

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | `set("k", "v", 60)` ثم `get("k")` | `"v"`, `hits=1` |
| 2 | `get("missing")` | `None`, `misses=1` |
| 3 | `set("k", "v", ttl=0)` ثم `get("k")` | `None` (انتهى فوراً) |
| 4 | `clear()` | `items_count=0` |
| 5 | `get_cache()` بـ `backend=redis` و Redis غير متاح | `MemoryCache` + warning |
| 6 | `stats()` | `backend`, `hits`, `misses`, `items_count` |
| 7 | Redis: `set(dict)` ثم `get` | نفس الـ dict |

---

## ⚠️ ملاحظات على التصميم

| النقطة | التأثير |
|--------|---------|
| **لا `delete(key)`** | يجب `clear()` لكل شيء — غير عملي |
| **لا `exists(key)`** | يمكن استنتاجه من `get` |
| **`json.dumps` فقط لـ dict/list** | `Decimal` يصبح string — لا يعود Decimal |
| **`str(value)` للأحادية** | `set(1.0842)` يعود `"1.0842"` string |
| **`flushdb()` في `clear()`** | خطير لو Redis مشترك مع تطبيقات أخرى |
| **`ping()` في `__init__`** | يبطئ الإنشاء (~ms) — لكنه ضروري |
| **لا thread safety** | `MemoryCache` غير آمن في multi-threading |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **YAGNI** | 4 دوال فقط |
| **Fail Soft** | Redis فشل → Memory |
| **Lazy Import** | `redis` فقط عند الحاجة |
| **Lazy Deletion** | حذف عند `get` فقط |
| **Di/Minimal** | Factory pattern |
| **Config-Driven** | `config.json` يقرر |
| **Serialization** | JSON للقواميس |
| **Stats** | hits/misses للـ Hit Ratio |
| **Zero Downtime** | Fallback تلقائي |

---

## 🗺️ Sprint 5

```mermaid
graph LR
    A["core/cache.py 🆕"] --> B["ForexService + Cache"]
    B --> C["Config: cache section"]
    C --> D["CLI: forex cache stats/clear"]
    D --> E["tests/test_cache.py"]

    style A fill:#bfb,stroke:#333,stroke-width:3px
    style B fill:#ddd,stroke:#333
```

---

## 📋 الخطوة التالية

> **دمج الكاش في `ForexService`**:
> - `get_current_price` → `cache.get(f"price:{pair}")`
> - `fetch_candles` → `cache.get(f"candles:{pair}:{interval}:{limit}")`
> - TTL مختلف: **السعر 30s**، **الشموع 5min**

**السؤال الجوهري:** ما الذي نُخزّنه في الكاش؟

**الجواب:**
1. **نتائج API** (السعر، الشموع) — يوفر quota.
2. **نتائج `analyze_pair`** (المؤشرات) — حساب مكلف.
3. **ليس** نتائج DB (SQLite أصلاً سريعة).

**تحدٍّ:** `Candle` و `Decimal` غير JSON-serializable → يجب تحويلها أو استخدام `to_dict`.

**الحل المقترح:** `Candle.to_dict()` قبل `cache.set`، `Candle.from_dict` بعد `cache.get`.

In [1]:
import time
from core.cache import MemoryCache

cache = MemoryCache(default_ttl=2)

# 1. فحص Miss أولاً
print('Initial Get (expect None):', cache.get('EUR/USD:price'))

# 2. تعيين قيمة وفحص Hit
cache.set('EUR/USD:price', '1.0850', ttl=1)
print('Immediate Get (expect 1.0850):', cache.get('EUR/USD:price'))

# 3. فحص بعد انتهاء الـ TTL
print('Waiting 1.2 seconds for expiration...')
time.sleep(1.2)
print('Expired Get (expect None):', cache.get('EUR/USD:price'))

# 4. طباعة الإحصائيات
print('Cache Stats:', cache.stats)

Initial Get (expect None): None
Immediate Get (expect 1.0850): 1.0850
Waiting 1.2 seconds for expiration...
Expired Get (expect None): None
Cache Stats: <bound method MemoryCache.stats of <core.cache.MemoryCache object at 0x0000023221C705F0>>


# 🔄 التغييرات فقط — دمج الكاش في `ForexService` (Sprint 5، الخطوة 2)

---

## 1️⃣ الاستيرادات الجديدة

```python
from core.cache import BaseCache, get_cache
```

---

## 2️⃣ `__init__` — حقن الكاش

```python
def __init__(
    self,
    source: TwelveDataSource | None = None,
    storage: Storage | None = None,
    cache: BaseCache | None = None,          # ← جديد
    rate_limit_delay: float = 1.0,
) -> None:
    self.source = source or TwelveDataSource()
    self.storage = storage or SQLiteStorage()
    self.cache = cache or get_cache()        # ← جديد
    self.rate_limit_delay = rate_limit_delay
```

---

## 3️⃣ `get_current_price` — نمط Cache-Aside

```python
def get_current_price(self, pair: str = "EUR/USD") -> Decimal:
    validated_pair = CurrencyPair.from_string(pair)
    cache_key = f"price:{validated_pair.symbol}"

    # 1. فحص الكاش أولاً
    cached_price = self.cache.get(cache_key)
    if cached_price is not None:
        logger.info(f"⚡ [Cache Hit] {validated_pair.symbol}: {cached_price}")
        return Decimal(str(cached_price))

    # 2. Cache Miss → API
    logger.info(f"🌐 [Cache Miss] جلب {validated_pair.symbol} من TwelveData...")
    candles = self.source.fetch_candles(pair=validated_pair.symbol, limit=1)
    if not candles:
        raise ValueError(f"تعذر جلب السعر لزوج {validated_pair.symbol}")

    latest_price = candles[0].close

    # 3. حفظ في الكاش (60 ثانية)
    self.cache.set(cache_key, str(latest_price), ttl=60)
    return latest_price
```

---

## 4️⃣ دوال جديدة في نهاية الكلاس

```python
def get_cache_stats(self) -> dict[str, Any]:
    """إرجاع إحصائيات الكاش الحالية."""
    return self.cache.stats()

def clear_cache(self) -> None:
    """مسح بيانات الكاش بالكامل."""
    self.cache.clear()
    logger.info("ForexService cache cleared.")
```

---

## 📋 ملخص التغييرات

| # | الموقع | التغيير |
|---|--------|---------|
| 1 | imports | + `BaseCache`, `get_cache` |
| 2 | `__init__` | + parameter `cache` + `self.cache` |
| 3 | `get_current_price` | Cache-Aside (get → miss → set) |
| 4 | دوال جديدة | `get_cache_stats`, `clear_cache` |

**🎯 ملاحظة:** `fetch_and_store`, `get_history`, `analyze_pair` **لم تتغير** — الكاش على `price` فقط في هذه الخطوة.

In [2]:
from services.forex_service import ForexService

service = ForexService()

print('--- الطلب الأول ---')
price1 = service.get_current_price('EUR/USD')
print('السعر:', price1)

print('\n--- الطلب الثاني مباشرة ---')
price2 = service.get_current_price('EUR/USD')
print('السعر:', price2)

print('\n--- إحصائيات الكاش ---')
print(service.get_cache_stats())


[2026-11-06 00:41:54] [INFO] [forex_tracker]: Connected to SQLite database at: ./data\candles.db
[2026-11-06 00:41:55] [WARNING] [forex_tracker]: Redis is not available (unknown command 'HELLO'). Falling back to MemoryCache.
[2026-11-06 00:41:55] [INFO] [forex_tracker]: 🌐 [Cache Miss] جلب السعر من TwelveData لزوج EUR/USD...
[2026-11-06 00:41:55] [INFO] [forex_tracker]: جلب 1 شمعة لزوج EUR/USD بإطار 1day من TwelveData...
[2026-11-06 00:41:55] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series


--- الطلب الأول ---


[2026-11-06 00:41:56] [INFO] [forex_tracker]: تم بنجاح استخراج 1 شمعة لزوج EUR/USD.
[2026-11-06 00:41:56] [INFO] [forex_tracker]: ⚡ [Cache Hit] السعر موجود في الكاش لزوج EUR/USD: 1.1222


السعر: 1.1222

--- الطلب الثاني مباشرة ---
السعر: 1.1222

--- إحصائيات الكاش ---
{'backend': 'memory', 'hits': 1, 'misses': 1, 'items_count': 1}


# 🖥️ التغييرات فقط — أوامر الكاش في CLI (Sprint 5، الخطوة 3)

---

## 1️⃣ `cli/commands.py` — تحديث `cmd_status`

```python
def cmd_status(args: argparse.Namespace) -> None:
    """عرض الحالة التشغيلية للنظام وإحصائيات الكاش."""
    config = Config()
    service = ForexService()
    cache_stats = service.get_cache_stats()

    print("\n--- Forex Tracker Status ---")
    print(f"App Name     : {config.get('app_name', 'Forex Tracker')}")
    print(f"Version      : {config.get('version', '0.3.0')}")
    print(f"Data Dir     : {config.get('data_dir', './data')}")
    print(f"Log File     : {config.get('log_file', './logs/app.log')}")
    print(f"Status       : Ready")

    print("\n--- Cache Performance ---")
    print(f"Backend      : {cache_stats.get('backend', 'memory').upper()}")
    print(f"Hits         : {cache_stats.get('hits', 0)}")
    print(f"Misses       : {cache_stats.get('misses', 0)}")
    print(f"Items Cached : {cache_stats.get('items_count', 0)}")
    print()
```

---

## 2️⃣ `cli/commands.py` — دالة جديدة

```python
def cmd_cache_clear(args: argparse.Namespace) -> None:
    """تفريغ بيانات الكاش بالكامل."""
    service = ForexService()
    try:
        service.clear_cache()
        print("\n🧹 Cache has been successfully cleared.\n")
    except ForexError as err:
        logger.error(f"فشل مسح الكاش: {err}")
        print(f"Error: {err}", file=sys.stderr)
        sys.exit(1)
```

---

## 3️⃣ `cli/parser.py` — import

```python
from cli.commands import (
    cmd_version, cmd_status, cmd_price, cmd_fetch, cmd_list,
    cmd_watchlist, cmd_analyze,
    cmd_alert_add, cmd_alert_list, cmd_alert_check, cmd_alert_delete,
    cmd_cache_clear,   # ← جديد
)
```

---

## 4️⃣ `cli/parser.py` — تسجيل أمر `cache`

```python
# 9. أمر cache
p_cache = subparsers.add_parser("cache", help="Manage application cache layer")
cache_sub = p_cache.add_subparsers(
    dest="cache_command",
    title="Cache Operations",
    description="Available cache operations"
)

p_cache_clear = cache_sub.add_parser("clear", help="Clear all cached market data")
p_cache_clear.set_defaults(func=cmd_cache_clear)
```

---

## 📋 ملخص التغييرات

| # | الملف | التغيير |
|---|-------|---------|
| 1 | `cli/commands.py` | `cmd_status` + قسم Cache Performance |
| 2 | `cli/commands.py` | + `cmd_cache_clear` |
| 3 | `cli/parser.py` | + import `cmd_cache_clear` |
| 4 | `cli/parser.py` | + subparser `cache` → `clear` |

---

## 🧪 التحقق

```bash
forex price --pair EUR/USD   # Miss
forex price --pair EUR/USD   # Hit
forex status                 # Hits=1, Misses=1, Items=1
forex cache clear            # 🧹
forex status                 # Items=0
```

**🎯 ملاحظة:** `cmd_status` الآن ينشئ `ForexService()` — يستهلك اتصالاً جديداً بـ SQLite + Cache. مقبول للـ CLI، لكن للأداء يمكن حقن service مشترك.